In [29]:
import json

In [34]:
import json
import unicodedata
from pathlib import Path
from difflib import SequenceMatcher
import re
from typing import List, Tuple
import os
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Any
from typing import List, Tuple
from collections import defaultdict
# Regex for ATT&CK technique IDs e.g. T1566 or T1566.001
ATTACK_ID_RE = re.compile(
    r"\b(?:T\d{4}(?:\.\d{3})?|TA\d{4})\b",
    flags=re.IGNORECASE
)
SENTENCE_SPLIT_RE = re.compile(r'(?<=[\.\?\!])\s+')

def normalize_text(s: str) -> str:
    if s is None:
        return ""
    try:
        s = unicodedata.normalize("NFKC", s)
    except Exception:
        pass
    return s.lower()


def find_id_sentences(text: str, valid_ids: set = None):
    """Return sentences containing ATT&CK IDs (validated)."""
    sentences = split_sentences(text)
    hits = []
    for sent in sentences:
        for m in ATTACK_ID_RE.finditer(sent):
            tid = m.group(0).upper()
            if valid_ids and tid not in valid_ids:
                continue
            hits.append({
                "id": tid,
                "sentence": sent.strip()
            })
    return hits


def find_name_sentences(text_norm: str, id_to_name: dict):
    """Return sentences containing ATT&CK technique names."""
    sentences = split_sentences(text_norm)
    hits = []
    for tid, name in id_to_name.items():
        name_norm = normalize_text(name)
        for sent in sentences:
            if name_norm in sent:
                hits.append({
                    "id": tid,
                    "name": name,
                    "sentence": sent.strip()
                })
    return hits
def _merge_fragmented_lines(lines: List[str], max_buffer_words: int = 200) -> List[str]:
    """
    Merge short/fragmented lines into larger lines by collecting successive non-empty lines
    until we encounter a line that ends with sentence punctuation or an explicit blank line.
    This helps with text which has lots of single-word lines.
    """
    merged = []
    buf = []
    for raw in lines:
        line = (raw or "").strip()
        if not line:
            # flush on blank lines
            if buf:
                merged.append(" ".join(buf).strip())
                buf = []
            continue

        buf.append(line)

        # heuristics to flush: if current line ends with punctuation or buffer is getting long
        if line.endswith((".", "?", "!", ":", ";")) or len(buf) >= max_buffer_words:
            merged.append(" ".join(buf).strip())
            buf = []

    if buf:
        merged.append(" ".join(buf).strip())
    return merged

def split_sentences(text: str) -> List[str]:
    """Return a list of sentences from text (keeps short fragments too)."""
    # Normalize newlines to spaces so sentences spanning lines are preserved
    if not text:
        return []
    text = text.replace("\r\n", " ").replace("\n", " ")
    # split and strip
    sents = [s.strip() for s in SENTENCE_SPLIT_RE.split(text) if s.strip()]
    return sents

def extract_ids_from_sentence(sentence: str, valid_ids: set = None) -> List[str]:
    """Return list of valid ATT&CK IDs found in a sentence."""
    ids = [m.group(0).upper() for m in ATTACK_ID_RE.finditer(sentence)]
    if valid_ids:
        ids = [tid for tid in ids if tid in valid_ids]
    return ids

def _norm_sent(s: str) -> str:
    # normalize for matching (lowercase + collapse whitespace)
    return " ".join(s.lower().split())
def sanitize_report_text(text: str,
                         min_block_len: int = 1,
                         window_size: int = 6,
                         window_threshold: int = 2,
                         line_id_threshold: int = 3
                         ) -> Tuple[str, List[List[str]]]:
    """
    Robust sanitizer:
      1) merges fragmented lines into coherent lines
      2) removes consecutive blocks of lines where each line contains an ATT&CK ID
      3) removes single lines/sentences containing too many ATT&CK IDs
      4) fallback: removes 'clusters' of lines where ID density in a sliding window >= threshold

    Returns (cleaned_text, removed_blocks)
    """
    if not text:
        return "", []

    # split original lines (preserve order)
    orig_lines = text.splitlines()

    # Step 1: Merge fragmented lines to better detect sentences/tables
    merged_lines = _merge_fragmented_lines(orig_lines)

    cleaned_lines = []
    buffer_block: List[str] = []
    removed_blocks: List[List[str]] = []

    def flush_block():
        nonlocal buffer_block
        if buffer_block:
            if len(buffer_block) < min_block_len:
                cleaned_lines.extend(buffer_block)
            else:
                removed_blocks.append(buffer_block[:])
            buffer_block = []

    # Step 2: Consecutive-line detection
    for line in merged_lines:
        if ATTACK_ID_RE.search(line):
            buffer_block.append(line)
        else:
            flush_block()
            cleaned_lines.append(line)
    flush_block()

    # Step 3: One-sentence / one-line with many IDs
    # Look for lines with >= line_id_threshold matches
    final_lines = []
    extra_removed = []
    for line in cleaned_lines:
        matches = ATTACK_ID_RE.findall(line)
        if len(matches) >= line_id_threshold:
            extra_removed.append(line)
        else:
            final_lines.append(line)

    if extra_removed:
        removed_blocks.append(extra_removed)
    cleaned_lines = final_lines

    # If we already removed something, return early
    if removed_blocks:
        return "\n".join(cleaned_lines), removed_blocks

    # Step 4: Fallback cluster detection — sliding window
    n = len(merged_lines)
    id_counts = [1 if ATTACK_ID_RE.search(l) else 0 for l in merged_lines]
    to_remove = [False] * n

    window_sum = sum(id_counts[:min(window_size, n)])
    if n:
        if window_sum >= window_threshold:
            for j in range(0, min(window_size, n)):
                to_remove[j] = True
    for i in range(1, n):
        prev_idx = i - 1
        remove_idx = i + window_size - 1
        window_sum = window_sum - id_counts[prev_idx]
        if remove_idx < n:
            window_sum += id_counts[remove_idx]
        if window_sum >= window_threshold:
            for j in range(i, min(i + window_size, n)):
                to_remove[j] = True

    cur_block = []
    final_cleaned = []
    for idx, line in enumerate(merged_lines):
        if to_remove[idx]:
            cur_block.append(line)
        else:
            if cur_block:
                removed_blocks.append(cur_block[:])
                cur_block = []
            final_cleaned.append(line)
    if cur_block:
        removed_blocks.append(cur_block[:])

    if not removed_blocks:
        return "\n".join(merged_lines), []

    return "\n".join(final_cleaned), removed_blocks

def build_ttp_sentences_from_id_hits(
    id_hits: List[Dict[str, Any]],
    sanitized_text: str,
    VALID_IDS:str,
    removed_blocks: List[List[str]] = None

) -> Dict[str, List[str]]:
    """
    Build mapping {technique_id: [sentences]} strictly from the sanitized text.
    - If id_hits are provided with 'sentence', keep only those whose sentence
      occurs in the sanitized text (so nothing from flushed blocks leaks in).
    - Otherwise, fallback to scanning the sanitized text by sentence.
    """
    ttp_sentences = defaultdict(list)

    # sentences from sanitized text (ground truth surface)
    sanitized_sents = [s.strip() for s in split_sentences(sanitized_text) if s.strip()]
    sanitized_sents_norm = {_norm_sent(s) for s in sanitized_sents}
    sanitized_lookup = { _norm_sent(s): s for s in sanitized_sents }  # norm -> original

    # path A: trust id_hits only if their sentence exists in sanitized text
    if id_hits and all(('id' in h and ('sentence' in h or 'sent' in h)) for h in id_hits):
        for h in id_hits:
            tid = h.get("id")
            sent = (h.get("sentence") or h.get("sent") or "").strip()
            if not tid or not sent:
                continue
            key = _norm_sent(sent)
            if key in sanitized_sents_norm:              # only keep if present post-sanitization
                ttp_sentences[tid].append(sanitized_lookup[key])
        # if we got anything valid, return it
        if ttp_sentences:
            return {k: v for k, v in ttp_sentences.items()}

    # path B: fallback — scan sanitized text sentences directly
    for s in sanitized_sents:
        for tid in extract_ids_from_sentence(s, VALID_IDS):
            ttp_sentences[tid].append(s)

    return {k: v for k, v in ttp_sentences.items()}


In [37]:
import json
import re


from pathlib import Path

def load_mapping():
    mapping_path = Path("/home/simonettos/thijs/data_augmentatio_stefano/files_with_ids_mitre_reports/tec_prefixed.json")
    if not mapping_path.exists():
        raise FileNotFoundError(f"Mapping file not found: {mapping_path}")

    id_to_name = json.loads(mapping_path.read_text(encoding="utf-8"))
    valid_ids = set(id_to_name.keys())
    return id_to_name, valid_ids


def slugify(s: str, max_len: int = 60) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"[^\w\s-]", "", s)
    s = re.sub(r"[\s_-]+", "_", s)
    s = s.strip("_")
    return s[:max_len] if len(s) > max_len else s

def make_out_name(item: dict, idx: int) -> str:
    """
    Build a safe-ish filename for each output.
    Adds idx to avoid collisions.
    """
    tid = (item.get("technique_id") or "unknown").strip().upper()
    src = slugify(item.get("source_name") or "source", max_len=50)
    hsh = (item.get("download_hash") or "")[:10]
    if not hsh:
        hsh = f"idx{idx}"
    return f"{tid}__{src}__{hsh}__{idx}.attack.json"

def process_item(item: dict, id_to_name: dict, VALID_IDS: set) -> dict:
    extracted_text = item.get("extracted_text") or ""
    technique_id = (item.get("technique_id") or "").strip().upper()

    sanitized_txt, removed_blocks = sanitize_report_text(extracted_text, min_block_len=1)

    # IDs from sanitized text (validated by VALID_IDS)
    id_hits = find_id_sentences(sanitized_txt, VALID_IDS)

    # Names: your helper expects *normalized text*
    sanitized_norm = normalize_text(sanitized_txt)
    name_hits = find_name_sentences(sanitized_norm, id_to_name)

    id_list = sorted({h.get("id") for h in id_hits if h.get("id")})
    name_list = sorted({h.get("name") for h in name_hits if h.get("name")})

    # Force the technique_id into ID_list (even if not detected / not in mapping)
    if technique_id:
        id_list = sorted(set(id_list) | {technique_id})

    ttp_sentences = build_ttp_sentences_from_id_hits(
        id_hits=id_hits,
        sanitized_text=sanitized_txt,
        VALID_IDS=VALID_IDS,
        removed_blocks=removed_blocks
    )

    return {
        "original_txt": extracted_text,
        "sanitized_txt": sanitized_txt,
        "ID_list": id_list,
        "Name_list": name_list,
        "TTP_sentences": ttp_sentences
    }

def main(INPUT_JSON: Path, out_dir: Path = Path("mitre_rep")):
    out_dir.mkdir(parents=True, exist_ok=True)

    id_to_name, VALID_IDS = load_mapping()

    data = json.loads(INPUT_JSON.read_text(encoding="utf-8", errors="replace"))
    if not isinstance(data, list):
        raise ValueError(f"Expected a list at root of JSON, got: {type(data)}")

    written = 0
    skipped = 0

    for idx, item in enumerate(data):
        if not isinstance(item, dict):
            skipped += 1
            continue
        if "extracted_text" not in item:
            skipped += 1
            continue

        out_payload = process_item(item, id_to_name, VALID_IDS)

        out_name = make_out_name(item, idx)
        out_path = out_dir / out_name

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(out_payload, f, indent=2, ensure_ascii=False)

        written += 1

    print(f"✅ Done. Wrote {written} JSONs to: {out_dir.resolve()}. Skipped: {skipped}.")

if __name__ == "__main__":
    # Change this to your actual JSON file that contains the list of extracted refs
   INPUT_JSON = Path("/home/simonettos/thijs/data_augmentatio_stefano/files_with_ids_mitre_reports/external_links_to_ttps.json")
   main(INPUT_JSON, out_dir=Path("/home/simonettos/thijs/data_augmentatio_stefano/files_with_ids_mitre_reports/mitre_rep"))


✅ Done. Wrote 2040 JSONs to: /home/simonettos/thijs/data_augmentatio_stefano/files_with_ids_mitre_reports/mitre_rep. Skipped: 1327.
